# Checkpoint 02 — Assistente de Saúde e Bem-estar (IoB)
**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI  
**Tema:** Interpretação educacional de dados de dispositivos vestíveis (wearables).  
**Integrante:** Julia Correa e Souza de Altino — RM 564870.
**Tema da atividade anterior:** Saúde e bem-estar (IoB).

Baseado nas etapas do notebook `CP2_ia_generativa.ipynb` do professor. Use **Ambiente de execução > Executar todas**, exceto a célula opcional de conversa interativa. Os testes HF não concluídos por erro 402 ficam identificados; Gemini segue executável. Registre as observações após a execução real.

> **Segurança:** não informe dados de saúde reais nem inclua tokens no notebook. As orientações são educativas, sem diagnóstico.

## 0. Configuração — Google Colab e APIs

In [1]:
%pip -q install "huggingface_hub>=0.30" "google-genai>=1.0" "gradio>=5.0,<7" "fastapi>=0.110" "uvicorn>=0.29" "requests>=2.31"

In [6]:
import os
from huggingface_hub import InferenceClient
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:
    HF_TOKEN = os.getenv("HF_TOKEN")
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

print("HF_TOKEN:", "configurado" if HF_TOKEN else "NÃO CONFIGURADO")
print("GEMINI_API_KEY:", "configurada" if GEMINI_API_KEY else "NÃO CONFIGURADA")
assert GEMINI_API_KEY, "Cadastre GEMINI_API_KEY nos Secrets do Colab."

# Professor: Llama 3.1 8B. Gemini configurado conforme notebook CP2 do professor.
MODELO_HF = "meta-llama/Llama-3.1-8B-Instruct"
MODELO_GEMINI = "gemini-2.5-flash"
cliente_hf = InferenceClient(model=MODELO_HF, token=HF_TOKEN, provider="auto") if HF_TOKEN else None
cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)
print("Clientes configurados. A disponibilidade e eventuais cotas serão verificadas nas chamadas.")

HF_TOKEN: configurado
GEMINI_API_KEY: configurada
Clientes configurados. A disponibilidade e eventuais cotas serão verificadas nas chamadas.


In [3]:
# Instruções de sistema do assistente Vita.
SYSTEM_PROMPT = """Você é Vita, um assistente educacional de saúde e bem-estar com foco em Internet of Bodies (IoB) e wearables.
Seu escopo inclui leitura geral de indicadores de smartwatches e pulseiras inteligentes: passos, atividade física, sono, frequência cardíaca, qualidade e limitações das medições e cuidados com privacidade.
Responda sempre em português brasileiro, de modo acolhedor, objetivo e claro.
Não apresente diagnósticos, não prescreva medicamentos, não interprete medições como confirmação de doenças e não substitua profissionais de saúde.
Em caso de emergência, sintomas graves ou risco imediato, recomende atendimento de urgência; no Brasil, o SAMU é 192.
Não peça informações pessoais identificáveis nem dados clínicos reais. Lembre que métricas de wearables podem conter imprecisões.
Se a pergunta não for sobre IoB, wearables, saúde ou bem-estar, responda educadamente que seu escopo está limitado a esses temas e sugira uma pergunta pertinente.
Quando útil, responda em no máximo 4 tópicos curtos; evite certezas indevidas."""
print("System prompt carregado:", len(SYSTEM_PROMPT), "caracteres")

System prompt carregado: 1031 caracteres


## Etapa 1 — Guardrails: escopo e formato

Comparação de três configurações de instruções: sem restrições, restrição temática e restrição de tema e formato. **Não declarar resultados HF como executados.**

In [12]:
def perguntar_hf(pergunta, system=SYSTEM_PROMPT, temperatura=0.7):
    if cliente_hf is None:
        raise RuntimeError("HF_TOKEN não cadastrado; teste Hugging Face indisponível.")
    resp = cliente_hf.chat_completion(
        messages=[{"role":"system","content":system},{"role":"user","content":pergunta}],
        temperature=temperatura, max_tokens=320)
    return resp.choices[0].message.content or "[Resposta vazia]"

versoes_system = {
    "Sem restrição": "Você é um assistente prestativo. Responda em português brasileiro.",
    "Restrição de tema": """Você é Vita, assistente de IoB, smartwatches, passos, sono e frequência cardíaca. Responda somente a perguntas sobre wearables e bem-estar. Recuse educadamente questões fora desse tema. Não diagnostique doenças nem prescreva tratamentos.""",
    "Restrição de tema e formato": SYSTEM_PROMPT + "\nOrganize todas as respostas em exatamente 3 tópicos numerados e concisos, incluindo a observação de que os dados não substituem avaliação profissional quando pertinente."
}
pergunta_dentro = "Um relógio inteligente registrou 7.000 passos ontem. Como posso entender essa métrica?"
pergunta_fora = "Me ensine a fazer uma lasanha à bolonhesa."
def perguntar_gemini(pergunta, system=SYSTEM_PROMPT, temperatura=0.7):
    resp = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI, contents=pergunta,
        config=types.GenerateContentConfig(system_instruction=system, temperature=temperatura, max_output_tokens=2048))
    return resp.text or "[Resposta sem texto; conferir bloqueio/limite de tokens]"


resultados_etapa1 = []
hf_indisponivel = False
for titulo, prompt in versoes_system.items():
    for rotulo, pergunta in [("Dentro", pergunta_dentro), ("Fora", pergunta_fora)]:
        if hf_indisponivel:
            print(f"[Hugging Face | {titulo}/{rotulo}] NÃO EXECUTADO: crédito indisponível neste ambiente.")
            continue
        try:
            resposta = perguntar_hf(pergunta, prompt, 0.3)
            resultados_etapa1.append((titulo, rotulo, resposta))
            print(f"\n[Hugging Face | {titulo} | {rotulo}]\n{resposta}\n")
        except Exception as erro:
            hf_indisponivel = True
            print(f"[Hugging Face] teste não concluído: {type(erro).__name__}: {str(erro)[:260]}")
            print("A execução prossegue com Gemini. O professor poderá testar o HF com sua própria chave e créditos.")

# Evidências executáveis com Gemini, mesmo se o Hugging Face não tiver créditos.
resultados_gemini_etapa1 = []
for titulo, prompt in versoes_system.items():
    for rotulo, pergunta in [("Dentro", pergunta_dentro), ("Fora", pergunta_fora)]:
        try:
            resposta = perguntar_gemini(pergunta, prompt, 0.3)
            resultados_gemini_etapa1.append((titulo, rotulo, resposta))
            print(f"\n[Gemini | {titulo} | {rotulo}]\n{resposta}\n")
        except Exception as erro:
            print(f"[Gemini | {titulo}/{rotulo}] Erro: {type(erro).__name__}: {str(erro)[:260]}")


[Hugging Face] teste não concluído: HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6ac79e6e-06be336d50255c503595f14b;753aa4a0-07d4-4881-b4f7-b049919ec48a)
For more information check: https://developer.mozilla.org/en-US/docs/Web
A execução prossegue com Gemini. O professor poderá testar o HF com sua própria chave e créditos.
[Hugging Face | Sem restrição/Fora] NÃO EXECUTADO: crédito indisponível neste ambiente.
[Hugging Face | Restrição de tema/Dentro] NÃO EXECUTADO: crédito indisponível neste ambiente.
[Hugging Face | Restrição de tema/Fora] NÃO EXECUTADO: crédito indisponível neste ambiente.
[Hugging Face | Restrição de tema e formato/Dentro] NÃO EXECUTADO: crédito indisponível neste ambiente.
[Hugging Face | Restrição de tema e formato/Fora] NÃO EXECUTADO: crédito indisponível neste ambiente.

[Gemini | Sem restrição | Dentro]
Que ótimo! 7.000 passos é um excelente ponto de partida e já indica um bom

### Observações do grupo — Etapa 1: Guardrails

- **Hugging Face:** a tentativa de execução retornou HTTP 402 (Payment Required), por falta de créditos de inferência. A implementação foi preservada para avaliação com credenciais habilitadas.
- **Gemini sem restrições:** respondeu sobre a contagem de 7.000 passos e também à pergunta fora do tema, sobre receita de lasanha.
- **Gemini com restrição temática:** respondeu à pergunta sobre passos e recusou a solicitação culinária, mantendo o foco em wearables e bem-estar.
- **Gemini com restrição de tema e formato:** em uma execução anterior houve respostas estruturadas e recusa da pergunta fora do tema; na última execução registrada neste notebook, ambas as chamadas retornaram HTTP 429 por limite de cota. Não foi possível repetir a verificação integral do formato nessa última execução.

**Conclusão:** os exemplos executados mostram o efeito dos guardrails sobre o escopo das respostas. As cotas dos serviços limitaram parte da validação.


## Etapa 2 — Comparação de temperatura (0.0 e 1.0)

Comparação das temperaturas 0.0 e 1.0 com perguntas equivalentes para os dois provedores.

In [10]:
perguntas_etapa2 = [
    "O que significam as fases do sono mostradas por um smartwatch e quais são as limitações dessa medição?",
    "Como analisar uma tendência de passos diários de uma pulseira inteligente sem tirar conclusões médicas?"
]
resultados_etapa2 = []
for pergunta in perguntas_etapa2:
    for modelo, funcao in [("Hugging Face", perguntar_hf),("Gemini",perguntar_gemini)]:
        if modelo == "Hugging Face" and hf_indisponivel:
            print("[Hugging Face] NÃO EXECUTADO: sem créditos; os testes de Gemini continuam.")
            continue
        for temperatura in [0.0,1.0]:
            try:
                resposta=funcao(pergunta,SYSTEM_PROMPT,temperatura)
                resultados_etapa2.append({"pergunta":pergunta,"modelo":modelo,"temperatura":temperatura,"resposta":resposta})
                print(f"\nMODELO: {modelo} | temperatura: {temperatura}\nPergunta: {pergunta}\nResposta: {resposta}\n")
            except Exception as erro:
                print(f"Falha em {modelo} com temperatura {temperatura}: {type(erro).__name__}: {erro}")
                if modelo == "Hugging Face":
                    hf_indisponivel = True
                    print("Sem créditos no Hugging Face; continuando com Gemini.")
                else:
                    print("Confira a chave, a cota ou o modelo Gemini antes de entregar.")

[Hugging Face] NÃO EXECUTADO: sem créditos; os testes de Gemini continuam.

MODELO: Gemini | temperatura: 0.0
Pergunta: O que significam as fases do sono mostradas por um smartwatch e quais são as limitações dessa medição?
Resposta: Olá! É ótimo que você esteja interessado em entender melhor as fases do sono monitoradas pelo seu smartwatch. Ele pode ser uma ferramenta interessante para o autoconhecimento


MODELO: Gemini | temperatura: 1.0
Pergunta: O que significam as fases do sono mostradas por um smartwatch e quais são as limitações dessa medição?
Resposta: Olá! É ótimo que você esteja interessado nas informações que seu smartwatch pode

[Hugging Face] NÃO EXECUTADO: sem créditos; os testes de Gemini continuam.

MODELO: Gemini | temperatura: 0.0
Pergunta: Como analisar uma tendência de passos diários de uma pulseira inteligente sem tirar conclusões médicas?
Resposta: Olá! É ótimo que você queira analisar seus dados de forma consciente e responsável


MODELO: Gemini | temperatura: 1.

### Observações do grupo — Etapa 2: Temperaturas 0.0 e 1.0

- **Hugging Face:** a comparação não pôde ser executada devido à ausência de créditos (HTTP 402).
- **Gemini:** foram realizadas quatro chamadas, com as mesmas duas perguntas em temperaturas 0.0 e 1.0.
- **Comparação observada:** os inícios das respostas mantiveram o contexto educacional, com pequenas diferenças de vocabulário e formulação. As respostas registradas foram curtas ou interrompidas, o que impede concluir se houve diferenças significativas de criatividade, profundidade ou precisão.
- **Limitação:** posteriormente, o limite `max_output_tokens` foi aumentado para 2048 na função de geração. A execução da Etapa 2 registrada neste arquivo ainda corresponde às saídas curtas anteriores; não se atribuem resultados não observados à nova configuração.

**Conclusão:** o uso das duas temperaturas foi demonstrado no Gemini, mas a comparação qualitativa ficou limitada pelas respostas registradas.


## Etapa 3 — Chat com memória compartilhada e troca de modelo

O código suporta mudança de HF para Gemini no mesmo histórico. Sem créditos HF, é realizado um teste de memória apenas com Gemini e a troca entre provedores permanece **não validada** nesta conta.

In [ ]:
def responder_chat(historico, modelo="hf", temperatura=0.4):
    if modelo == "hf":
        if cliente_hf is None:
            raise RuntimeError("HF_TOKEN não configurado")
        resposta=cliente_hf.chat_completion(messages=historico,temperature=temperatura,max_tokens=340)
        return resposta.choices[0].message.content or "[Sem texto]"
    if modelo != "gemini":
        raise ValueError("Modelo deve ser hf ou gemini")
    conteudos=[]
    for item in historico:
        if item["role"] == "system":
            continue
        role="model" if item["role"] == "assistant" else "user"
        conteudos.append(types.Content(role=role,parts=[types.Part.from_text(text=item["content"])]))
    resposta=cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,contents=conteudos,
        config=types.GenerateContentConfig(system_instruction=historico[0]["content"],temperature=temperatura,max_output_tokens=450))
    return resposta.text or "[Sem texto]"

def novo_historico():
    return [{"role":"system","content":SYSTEM_PROMPT}]

def enviar_mensagem(historico, texto, modelo, temperatura=0.4):
    candidato=historico+[{'role':'user','content':texto}]
    resposta=responder_chat(candidato,modelo,temperatura)
    return candidato+[{'role':'assistant','content':resposta}],resposta

# Teste reproduzível de troca entre modelos, sem digitação manual.
historico_teste=novo_historico()
if hf_indisponivel:
    print("Teste de troca HF → Gemini não executado por falta de créditos HF.")
    historico_teste,r1=enviar_mensagem(historico_teste,"Meu objetivo fictício desta semana é caminhar 6.000 passos por dia. Repita esse objetivo.","gemini")
    print("[Gemini - primeira mensagem]", r1)
else:
    try:
        historico_teste,r1=enviar_mensagem(historico_teste,"Meu objetivo fictício desta semana é caminhar 6.000 passos por dia. Repita esse objetivo.","hf")
        print("[HF]",r1)
    except Exception as erro:
        hf_indisponivel=True
        print("HF indisponível durante troca:",type(erro).__name__)
        historico_teste,r1=enviar_mensagem(historico_teste,"Meu objetivo fictício desta semana é caminhar 6.000 passos por dia. Repita esse objetivo.","gemini")
historico_teste,r2=enviar_mensagem(historico_teste,"Qual objetivo de passos eu mencionei antes?","gemini")
print("[Gemini após troca]",r2)
print("Mensagens no histórico (incluindo system):",len(historico_teste))

In [ ]:
# Conversa interativa por terminal (execução opcional).
def chat_interativo():
    historico=novo_historico()
    modelo="hf"
    print("Comandos: sair | limpar | historico | modelo hf | modelo gemini")
    while True:
        texto=input(f"[{modelo}] Você: ").strip()
        if texto.lower()=="sair": break
        if texto.lower()=="limpar":
            historico=novo_historico();print("Histórico limpo.");continue
        if texto.lower()=="historico":
            for m in historico: print(m['role'],':',m['content'][:140]);continue
        if texto.lower() in ("modelo hf","modelo gemini"):
            modelo="hf" if texto.lower()=="modelo hf" else "gemini"
            print("Modelo alterado para",modelo);continue
        if texto:
            historico,resposta=enviar_mensagem(historico,texto,modelo)
            print("Assistente:",resposta)
# chat_interativo()  # remova o comentário para executar o chat manual.

### Observações do grupo — Etapa 3: Memória e troca de modelos

- O código mantém o histórico de mensagens e implementa a seleção entre Hugging Face e Gemini.
- **Não foi possível validar a troca efetiva entre provedores**, pois a conta Hugging Face retornou HTTP 402.
- O teste automático de memória desta etapa **não foi executado no notebook entregue**, em razão das limitações de cota do Gemini. Portanto, a recuperação do contexto permanece implementada, mas não comprovada por uma saída real nesta entrega.


## Etapa 4 — Gradio: seletor de modelo e temperatura

O seletor mantém **Hugging Face** e **Gemini**, como solicitado. No ambiente atual, escolha Gemini para registrar o print e testar a interface. A opção HF continua implementada para testes com credenciais e créditos válidos.

In [16]:
import gradio as gr

def texto_da_mensagem(conteudo):
    if isinstance(conteudo,str):return conteudo
    if isinstance(conteudo,list):
        return " ".join(str(item.get("text","")) for item in conteudo if isinstance(item,dict))
    return str(conteudo)

def responder_gradio(message,history,modelo,temperatura):
    mensagens=novo_historico()
    for item in history:
        if isinstance(item,dict) and item.get("role") in ("user","assistant"):
            mensagens.append({"role":item["role"],"content":texto_da_mensagem(item.get("content",""))})
    mensagens.append({"role":"user","content":str(message)})
    try:
        return responder_chat(mensagens,"hf" if modelo=="Hugging Face" else "gemini",float(temperatura))
    except Exception as erro:
        if modelo == "Hugging Face":
            return "Hugging Face não disponível nesta conta (possível falta de créditos, erro 402). O professor pode testar com sua chave e créditos. Detalhe: " + str(erro)[:180]
        return "Falha no Gemini. Verifique chave, cota e modelo. Detalhe: " + str(erro)[:180]

interface=gr.ChatInterface(
    fn=responder_gradio,
    title="Vita | Assistente de Saúde e Bem-estar (IoB)",
    description="Entenda passos, sono e indicadores de wearables. Conteúdo educacional: não oferece diagnóstico ou prescrição.",
    additional_inputs=[
        gr.Radio(choices=["Hugging Face","Gemini"],value="Gemini",label="Modelo de IA"),
        gr.Slider(minimum=0.0,maximum=1.0,value=0.3,step=0.1,label="Temperatura")
    ],
    examples=[
        ["O que meu smartwatch mede durante o sono?","Gemini",0.3],
        ["Quais limitações existem na leitura de frequência cardíaca pelo relógio?","Hugging Face",0.3],
        ["Como interpretar uma queda de passos durante a semana?","Gemini",0.9]
    ]
)
print("Interface Gradio configurada.")

Interface preparada. Execute a próxima célula para abrir.


In [17]:
interface.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: [link temporário expirável]

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Observações do grupo — Etapa 4: Interface Gradio

- A interface foi **iniciada com sucesso** no Google Colab, com título, campo de chat, exemplos de perguntas, seletor Hugging Face/Gemini e slider de temperatura de 0.0 a 1.0.
- A evidência visual está em `prints/gradio.png`.
- **Limitação:** a captura comprova a abertura e os controles da interface, não a geração de mensagens no chat. As respostas via interface não foram validadas nesta execução devido às restrições de API.


## Etapa 5 (bônus) — API REST com sessões independentes
Implementação principal em `app_avaliacao.py` (arquivo incluído no projeto). O notebook inicia um servidor local e testa isolamento entre duas sessões.

In [ ]:
# Dependência da API REST: arquivo app_avaliacao.py no diretório de execução.
import pathlib
assert pathlib.Path("app_avaliacao.py").exists(), "Arquivo app_avaliacao.py não encontrado para a pasta de arquivos do Colab antes de executar o bônus."
import subprocess, sys, time
ambiente=os.environ.copy()
ambiente["GEMINI_API_KEY"]=GEMINI_API_KEY
ambiente["SYSTEM_PROMPT"]=SYSTEM_PROMPT
ambiente["MODELO_GEMINI"]=MODELO_GEMINI
servidor=subprocess.Popen([sys.executable,"-m","uvicorn","app_avaliacao:app","--host","127.0.0.1","--port","8000"],env=ambiente)
time.sleep(4)
assert servidor.poll() is None, "Servidor não iniciou. Confira logs e se a porta 8000 está livre."
print("Servidor ativo em http://127.0.0.1:8000")

In [ ]:
import requests
try:
    raiz="http://127.0.0.1:8000"
    health=requests.get(raiz+"/health",timeout=15)
    health.raise_for_status()
    print("Health:",health.json())
    pedidos=[
        ("sessao_a","Meu nome fictício é Ana. Lembre desse nome para a próxima resposta."),
        ("sessao_a","Que nome eu informei?"),
        ("sessao_b","Qual é o meu nome? Responda apenas se eu tiver informado antes nesta conversa.")
    ]
    for sessao,mensagem in pedidos:
        resposta=requests.post(raiz+"/chat",json={"session_id":sessao,"mensagem":mensagem,"modelo":"gemini","temperatura":0.2},timeout=90)
        resposta.raise_for_status()
        print(sessao, "=>", resposta.json()["resposta"])
    print("Sessões ativas:",requests.get(raiz+"/sessions",timeout=10).json())
finally:
    servidor.terminate()
    try: servidor.wait(timeout=8)
    except subprocess.TimeoutExpired: servidor.kill()
    print("Servidor encerrado")

### Observações do grupo — Etapa 5 (bônus): API REST

- A implementação está no arquivo `app_avaliacao.py`, com endpoints para saúde da API, conversas e sessões independentes.
- As células de inicialização e teste do servidor **não foram executadas neste notebook entregue**. Portanto, o isolamento entre sessões ainda não foi comprovado por chamadas reais nesta entrega.
- A implementação utiliza históricos separados por `session_id`, armazenados temporariamente em memória; a persistência entre reinicializações não está contemplada.


## Síntese dos requisitos

- [x] Identificação da integrante e tema.
- [x] Implementação de Hugging Face e Gemini preservada.
- [x] Testes reais de guardrails com Gemini e limitações registradas.
- [x] Testes de temperatura com Gemini, com ressalvas sobre respostas curtas.
- [x] Código de memória e alternância disponível (não validado integralmente).
- [x] Interface Gradio iniciada e captura em `prints/gradio.png`.
- [x] Código da API bônus incluído (não executado nesta entrega).
- [x] Notebook com saídas reais preservadas.
- [ ] Publicar em repositório GitHub público e entregar somente o link.
